# fyM1 00 — base-model baseline (T4)
One candidate per run. **Runtime → Run all**; the notebook picks the first candidate in `CANDIDATES` whose
results file on `Fan1000-Prog/fyM1-ckpt` (`results/baseline/<slug>.json`) is not `done`, and skips sections
already saved there, so a rerun after a disconnect resumes. Restart the runtime between candidates
(**Runtime → Restart session**, then **Run all**). Set `FORCE` to run one candidate explicitly.

Measures, all 4-bit: Belebele `plt_Latn` (900, 0-shot, next-token over 1–4), FLORES+ `plt_Latn`↔`eng_Latn`
devtest first 200 chrF++ (greedy), tokens per Malagasy word (1,000 Wikipedia sentences), 20 greeting answers,
and a short QLoRA smoke train (seq 2048, bs 1 × grad-accum 8) for peak VRAM and sec/step.
Needs Colab secret `HF_TOKEN` with notebook access, and the repo `src/` uploaded to `fyM1-ckpt` under `code/src/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U unsloth unsloth_zoo sacrebleu

In [ ]:
CANDIDATES = {
    # slug: (hub id, QLoRA smoke steps, push 4-bit weights to fyM1-ckpt base-4bit/<slug>/)
    # 4-bit push off: on T4 Unsloth keeps gemma4 in fp32 and save_pretrained stalled at ~11.5/12.7 GB CPU RAM
    "gemma-4-12b-it": ("google/gemma-4-12B-it", 30, False),
    "afriqueqwen-8b": ("McGill-NLP/AfriqueQwen-8B", 15, False),
    "gemma-3-12b-it": ("unsloth/gemma-3-12b-it-unsloth-bnb-4bit", 15, False),
}
FORCE = None  # e.g. "afriqueqwen-8b"
REPO = "Fan1000-Prog/fyM1-ckpt"
BELEBELE_BATCH, GEN_BATCH, FLORES_N, TPW_SENTENCES = 4, 8, 200, 1000
FALLBACK_GREETINGS = ["Sali ahn", "De ahoana?", "Manao ahoana ianao?", "De aona ry zalahy?", "Salut, mbola tsy natory ve?"]

In [ ]:
import os, sys, json, time, datetime
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.errors import EntryNotFoundError

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never print it
api = HfApi()
code_dir = snapshot_download(REPO, allow_patterns=["code/**"], local_dir="/content/fym1code")
sys.path.insert(0, f"{code_dir}/code/src")
from fym1.eval import baseline as bl


def load_results(slug):
    try:
        return json.load(open(hf_hub_download(REPO, f"results/baseline/{slug}.json", force_download=True)))
    except EntryNotFoundError:
        return {}


def save_results(slug, results):
    path = f"/content/{slug}.json"
    with open(path, "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    api.upload_file(path_or_fileobj=path, path_in_repo=f"results/baseline/{slug}.json", repo_id=REPO,
                    commit_message=f"baseline {slug}: {', '.join(k for k in results if k not in ('model', 'env'))}")


SLUG = FORCE or next((s for s in CANDIDATES if not load_results(s).get("done")), None)
assert SLUG, "all candidates done"
HUB_ID, SMOKE_STEPS, PUSH_4BIT = CANDIDATES[SLUG]
R = load_results(SLUG)
R.setdefault("model", {"slug": SLUG, "hub_id": HUB_ID, "quant": "bnb 4-bit nf4 (Unsloth load_in_4bit)"})
print("CANDIDATE", SLUG, "| sections already saved:", [k for k in R if k not in ("model", "env")])

In [ ]:
import torch
from unsloth import FastModel

def section(name):
    print(f"== {name}: {'skip (saved)' if name in R else 'run'}")
    return name not in R

t0 = time.time()
try:
    model, processor = FastModel.from_pretrained(HUB_ID, max_seq_length=2048, load_in_4bit=True, full_finetuning=False)
except Exception as e:  # gated/403 or unsupported architecture: record and move on
    R["skipped"] = f"{type(e).__name__}: {str(e)[:300]}"
    R["done"] = True
    save_results(SLUG, R)
    raise
tok = getattr(processor, "tokenizer", processor)
tok.padding_side = "left"
R["env"] = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "model_dtype": str(model.dtype),
            "vram_after_load_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
            "load_s": round(time.time() - t0), "utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
print(R["env"], "vram after load GB:", round(torch.cuda.memory_allocated() / 2**30, 2))


def chat(prompt):
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)


def encode(texts):
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")


@torch.inference_mode()
def generate(prompts, max_new_tokens, sample):
    out = []
    for i in range(0, len(prompts), GEN_BATCH):
        enc = encode([chat(p) for p in prompts[i:i + GEN_BATCH]])
        kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
        ids = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        out += tok.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return out


@torch.inference_mode()
def last_logits(enc):
    try:
        return model(**enc, logits_to_keep=1).logits[:, -1]
    except TypeError:  # forward without logits_to_keep
        return model(**enc).logits[:, -1]

In [ ]:
if PUSH_4BIT and section("base_4bit"):
    t0 = time.time()
    d = f"/content/base-4bit/{SLUG}"
    try:
        model.save_pretrained(d)
        processor.save_pretrained(d)
        api.upload_folder(folder_path=d, path_in_repo=f"base-4bit/{SLUG}", repo_id=REPO,
                          commit_message=f"bnb-4bit weights of {HUB_ID}")
        R["base_4bit"] = {"path": f"base-4bit/{SLUG}", "upload_s": round(time.time() - t0)}
    except Exception as e:  # not fatal: evals still run
        R["base_4bit"] = {"error": f"{type(e).__name__}: {str(e)[:300]}"}
    save_results(SLUG, R)
    print(R["base_4bit"])

In [ ]:
from datasets import load_dataset

FastModel.for_inference(model)
if section("belebele"):
    t0 = time.time()
    rows = load_dataset("facebook/belebele", "plt_Latn", split="test")
    digit_ids = [tok.encode(d, add_special_tokens=False)[-1] for d in "1234"]
    preds, nan_rows = [], 0
    with torch.inference_mode():
        for i in range(0, len(rows), BELEBELE_BATCH):
            batch = rows[i:i + BELEBELE_BATCH]
            prompts = [bl.belebele_prompt({k: batch[k][j] for k in batch}) for j in range(len(batch["question"]))]
            logits = last_logits(encode([chat(p) for p in prompts]))[:, digit_ids].float()
            nan_rows += int(torch.isnan(logits).any(-1).sum())
            preds += (logits.argmax(-1) + 1).tolist()
    R["belebele"] = {"acc": round(bl.accuracy(preds, rows["correct_answer_num"]), 4), "n": len(preds),
                     "nan_rows": nan_rows, "seconds": round(time.time() - t0),
                     "protocol": "0-shot, chat template, English instruction, argmax of next-token logits over '1'-'4'"}
    save_results(SLUG, R)
print("BELEBELE", R["belebele"])

In [ ]:
if section("flores"):
    t0 = time.time()
    try:
        mg = load_dataset("openlanguagedata/flores_plus", "plt_Latn", split="devtest")
        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
        print("flores columns:", mg.column_names)
        mg_txt, en_txt = mg["text"][:FLORES_N], en["text"][:FLORES_N]
        if "id" in mg.column_names:
            assert mg["id"][:FLORES_N] == en["id"][:FLORES_N], "FLORES rows not aligned"
        hyp_mg = [bl.first_line(t) for t in generate([bl.translation_prompt(s, "English", "Malagasy") for s in en_txt], 256, False)]
        hyp_en = [bl.first_line(t) for t in generate([bl.translation_prompt(s, "Malagasy", "English") for s in mg_txt], 256, False)]
        R["flores"] = {"eng-plt_chrf++": round(bl.chrf_pp(hyp_mg, mg_txt), 2),
                       "plt-eng_chrf++": round(bl.chrf_pp(hyp_en, en_txt), 2), "n": FLORES_N,
                       "seconds": round(time.time() - t0),
                       "protocol": "devtest first 200, 0-shot chat prompt, greedy, max 256 new tokens, first line kept"}
        save_results(SLUG, R)
    except Exception as e:  # gated dataset not yet accepted: leave the section unsaved so a rerun retries it
        print("FLORES PENDING:", type(e).__name__, str(e)[:300])
print("FLORES", R.get("flores", "pending"))

In [ ]:
if section("tokens_per_word"):
    wiki = load_dataset("wikimedia/wikipedia", "20231101.mg", split="train").shuffle(seed=42)
    sents = bl.sample_sentences((r["text"] for r in wiki), n=TPW_SENTENCES)
    R["tokens_per_word"] = {"value": round(bl.tokens_per_word(sents, lambda s: len(tok.encode(s, add_special_tokens=False))), 3),
                            "n_sentences": len(sents), "source": "wikimedia/wikipedia 20231101.mg, shuffle seed 42, 5-60 words"}
    save_results(SLUG, R)
print("TPW", R["tokens_per_word"])

In [ ]:
if section("greetings"):
    try:
        rows = [json.loads(l) for l in open(hf_hub_download(REPO, "code/eval/prompts.jsonl"))]
        greetings = [r["prompt"] for r in rows if r["category"].startswith("greet")][:20]
        source = "eval/prompts.jsonl greetings, first 20"
    except EntryNotFoundError:
        greetings, source = FALLBACK_GREETINGS, "fallback made-up greetings"
    torch.manual_seed(42)
    answers = generate(greetings, 128, True)
    R["greetings"] = {"source": source, "sampling": "temp 0.7, top_p 0.9, max 128 new tokens, seed 42, no system prompt",
                      "answers": [{"prompt": p, "answer": a.strip()} for p, a in zip(greetings, answers)]}
    save_results(SLUG, R)
for a in R["greetings"]["answers"]:
    print(repr(a["prompt"]), "->", repr(a["answer"][:200]))

In [ ]:
if section("qlora_smoke"):
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    from datasets import Dataset

    wiki = load_dataset("wikimedia/wikipedia", "20231101.mg", split="train").shuffle(seed=7)
    chunks, buf = [], ""
    for r in wiki:  # ~2048-token chunks of Malagasy text: worst-case length for the fit test
        buf += r["text"] + "\n\n"
        if len(tok.encode(buf, add_special_tokens=False)) >= 2048:
            chunks.append(buf)
            buf = ""
            if len(chunks) == SMOKE_STEPS * 8:
                break
    multimodal = hasattr(model.config, "vision_config")
    lora = dict(r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=42, use_gradient_checkpointing="unsloth")
    if multimodal:
        model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                         finetune_attention_modules=True, finetune_mlp_modules=True, **lora)
    else:
        model = FastModel.get_peft_model(model, target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                                "gate_proj", "up_proj", "down_proj"], **lora)
    step_times = []

    class StepTimer(TrainerCallback):
        def on_step_end(self, args, state, control, **kw):
            step_times.append(time.time())

    tok.padding_side = "right"
    compute_dtype = next(p.dtype for p in model.parameters() if p.is_floating_point())  # Unsloth may force fp32 on T4
    trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=Dataset.from_dict({"text": chunks}),
                         callbacks=[StepTimer()],
                         args=SFTConfig(output_dir="/content/smoke_out", per_device_train_batch_size=1,
                                        gradient_accumulation_steps=8, max_steps=SMOKE_STEPS, learning_rate=2e-4,
                                        lr_scheduler_type="cosine", warmup_steps=3, logging_steps=1, optim="adamw_8bit",
                                        fp16=compute_dtype == torch.float16, bf16=False, max_length=2048,
                                        dataset_text_field="text",
                                        packing=False, save_strategy="no", report_to="none", seed=42))
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        trainer.train()
        losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
        gaps = [b - a for a, b in zip(step_times, step_times[1:])]
        R["qlora_smoke"] = {"ok": True, "steps": SMOKE_STEPS, "seq_len": 2048, "batch": "1 x grad-accum 8",
                            "peak_vram_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2),
                            "sec_per_step": round(sum(gaps) / len(gaps), 1) if gaps else None,
                            "seconds": round(time.time() - t0), "loss_first": losses[0], "loss_last": losses[-1],
                            "nan_loss": any(l != l for l in losses), "lora": "r16 a16 all linear (language only)",
                            "compute_dtype": str(compute_dtype)}
    except Exception as e:  # OOM or a training error is a result here, not a crash
        R["qlora_smoke"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:300]}", "steps": SMOKE_STEPS,
                            "steps_done": len(step_times), "compute_dtype": str(compute_dtype),
                            "peak_vram_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2)}
    save_results(SLUG, R)
print("QLORA", R["qlora_smoke"])

In [ ]:
R["done"] = True  # a missing "flores" section means pending: rerun with FORCE = SLUG to fill only that
save_results(SLUG, R)
print("BASELINE_DONE" if "flores" in R else "BASELINE_DONE_FLORES_PENDING", SLUG,
      json.dumps({k: R[k] for k in R if k not in ("greetings",)}, ensure_ascii=False))